# Проект 1. Локальные vs глобальные модели на похожих рядах (M4 Quarterly)

## 0. Импорты

In [1]:
# Imports
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from tslearn.clustering import TimeSeriesKMeans
import plotly.graph_objects as go

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoTheta
from utilsforecast.losses import smape

# Add src to path
SRC_PATH = Path("src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from config import SEED, rng, HORIZON, SEASON_LEN, N_SERIES
from datasets import load_m4_datasets, select_aligned_series, wide_to_long_train, wide_to_long_test
from preprocessing import normalize_inversable

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)

# Fix randomness
import random
random.seed(SEED)
np.random.seed(SEED)

/usr/local/lib/python3.11/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)
/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Загрузка и базовая предобработка данных

In [2]:
# Load data
train_wide, test_wide, info = load_m4_datasets()
print(f"Raw shapes:\n\t{train_wide.shape}, {test_wide.shape}, {info.shape}\n")

# Select aligned series (with the same start and same end)
train_wide, test_wide, info = select_aligned_series(N_SERIES, train_wide, test_wide, info, rng)
print(f"After choosing aligned series:\n\t{train_wide.shape}, {test_wide.shape}\n")

# Transform from wide to long format 
train_long = wide_to_long_train(train_wide, info)
test_long = wide_to_long_test(test_wide, train_long, info)
print(f"After transforming to long:\n\t{train_long.shape}, {test_long.shape}")

# Scale data
train_long_scaled, test_long_scaled, inverse_scale_df = normalize_inversable(train_long, test_long)

Raw shapes:
	(24000, 867), (24000, 9), (100000, 6)

Chosen aligned series group size: 298
Start and end dates: 1983-09-30 12:00:00, 2006Q1
After choosing aligned series:
	(200, 867), (200, 9)

After transforming to long:
	(16600, 3), (1600, 3)


### Примеры рядов

In [3]:
# Plot a few random series (train + test) on one interactive chart
sample_ids = rng.choice(train_long["id"].unique(), size=3, replace=False)

fig = go.Figure()
for sid in sample_ids:
    df_train = train_long[train_long["id"] == sid]
    df_test = test_long[test_long["id"] == sid]

    fig.add_trace(go.Scatter(x=df_train["ds"], y=df_train["y"], mode="lines", name=f"{sid} train"))
    fig.add_trace(go.Scatter(x=df_test["ds"], y=df_test["y"], mode="lines", name=f"{sid} test", line=dict(dash="dash")))

fig.update_layout(title="Series examples before normalization", xaxis_title="ds", yaxis_title="y")
fig.show()

# Plot the same series after normalization
fig = go.Figure()
for sid in sample_ids:
    df_train = train_long_scaled[train_long_scaled["id"] == sid]
    df_test = test_long_scaled[test_long_scaled["id"] == sid]
    fig.add_trace(go.Scatter(x=df_train["ds"], y=df_train["y"], mode="lines", name=f"{sid} train (scaled)"))
    fig.add_trace(go.Scatter(x=df_test["ds"], y=df_test["y"], mode="lines", name=f"{sid} test (scaled)", line=dict(dash="dash")))
fig.update_layout(title="Series examples after normalization", xaxis_title="ds", yaxis_title="y_scaled")
fig.show()



## 3. Кластеризация `TimeSeriesKMeans`

Строим кластеризацию на $k$ кластеров для нескольких значений $k$ ([2 ... 15]) и выбираем значение $k$ по методу локтя.

In [4]:
from clasterization import build_ts_matrix, plot_elbow_curve

series_ids, X_series, max_len = build_ts_matrix(train_long_scaled)
plot_elbow_curve(X_series, k_values=range(2, 30))

По методу локтя выбираем $k = 5$ (переход $k = 4 \to 5$ даёт прирост сильно больше, $5 \to 6$ и далее).

In [5]:
# Fit final clustering
OPTIMAL_CLUSTERS = 5
kmeans = TimeSeriesKMeans(
    n_clusters=OPTIMAL_CLUSTERS,
    metric="dtw",
    random_state=SEED,
    verbose=False,
)
labels = kmeans.fit_predict(X_series)

clusters_df = pd.DataFrame({"id": series_ids, "cluster": labels})
cluster_map = clusters_df.set_index("id")["cluster"].to_dict()
train_long["cluster"] = train_long["id"].map(cluster_map)
test_long["cluster"] = test_long["id"].map(cluster_map)
train_long_scaled["cluster"] = train_long_scaled["id"].map(cluster_map)
test_long_scaled["cluster"] = test_long_scaled["id"].map(cluster_map)

Примеры рядов в кластерах:

In [6]:
# Plot a few series per cluster (normalized)
for cluster_index in sorted(train_long_scaled["cluster"].unique()):
    ids = train_long_scaled.loc[train_long_scaled["cluster"] == cluster_index, "id"].unique()
    ids = ids[:10]

    fig = go.Figure()
    for sid in ids:
        df_series = train_long_scaled[train_long_scaled["id"] == sid]
        fig.add_trace(go.Scatter(x=df_series["ds"], y=df_series["y"], mode="lines", name=str(sid)))
    fig.update_layout(title=f"Cluster {cluster_index} (normalized)", xaxis_title="ds", yaxis_title="y_scaled")
    fig.show()


Мы видим, что ряды внутри кластеров действительно похожи друг на друга.

## 4. Бейзлайны (обязательные)

Базовые модели: Naive, SeasonalNaive, AutoTheta, AutoETS.

In [7]:
models = [
    Naive(),
    SeasonalNaive(season_length=SEASON_LEN),
    AutoTheta(season_length=SEASON_LEN),
    AutoETS(season_length=SEASON_LEN),
]

sf = StatsForecast(models=models, freq="Q", n_jobs=-1)

# Train on normalized series
sf.fit(df=train_long_scaled, id_col="id")
preds_base_scaled = sf.predict(h=HORIZON)

# Inverse scaling to original scale for evaluation/plots
base_models = ["Naive", "SeasonalNaive", "AutoTheta", "AutoETS"]
preds_base = inverse_scale_df(preds_base_scaled, base_models)

base_eval = test_long.merge(preds_base, on=["id", "ds"], how="inner")
smape_base = smape(base_eval, models=base_models, id_col="id", target_col="y")

# mean over series
summary_base = pd.DataFrame({
    "sMAPE": smape_base[base_models].mean(),
}).sort_values("sMAPE")

summary_base

,sMAPE
AutoETS,0.036463
AutoTheta,0.036586
Naive,0.050769
SeasonalNaive,0.061119


## 5. ML-модели: глобальная, локальная, и кластерная

### 5.1. Создаем признаки и обучаем 3 вида ML-моделей

In [8]:
from ml_models import make_ml_dataset, fit_model, forecast_recursive

# Datasets
feature_frame, target, feature_cols, categorical_feature_names = make_ml_dataset(train_long_scaled)

# Global model
global_model = fit_model(feature_frame, target, categorical_feature_names)

# Cluster models
cluster_models = {}
for cluster_index in sorted(train_long_scaled["cluster"].unique()):
    ids = clusters_df.loc[clusters_df["cluster"] == cluster_index, "id"]
    df_cluster = train_long_scaled[train_long_scaled["id"].isin(ids)]

    feature_frame_cluster, target_cluster, _, _ = make_ml_dataset(df_cluster)
    model = fit_model(feature_frame_cluster, target_cluster, categorical_feature_names)
    cluster_models[cluster_index] = model

# Local models
local_models = {}
for sid, df_series in train_long_scaled.groupby("id"):
    feature_frame_series, target_series, _, _ = make_ml_dataset(df_series)
    model = fit_model(feature_frame_series, target_series, categorical_feature_names)
    local_models[sid] = model


### 5.2. Делаем предсказания ML-моделями

In [9]:
rows = []
for sid, df_series in train_long_scaled.groupby("id"):
    test_ds = test_long_scaled.loc[test_long_scaled["id"] == sid, "ds"].values

    # Global
    pred_global = forecast_recursive(global_model, df_series, test_ds, feature_cols)

    # Cluster
    cluster_index = cluster_map[sid]
    pred_cluster = forecast_recursive(cluster_models[cluster_index], df_series, test_ds, feature_cols)

    # Local
    pred_local = forecast_recursive(local_models[sid], df_series, test_ds, feature_cols)

    for ds, p_global, p_cluster, p_local in zip(test_ds, pred_global, pred_cluster, pred_local):
        rows.append({
            "id": sid,
            "ds": ds,
            "GlobalML": p_global,
            "ClusterML": p_cluster,
            "LocalML": p_local,
        })

preds_ml_scaled = pd.DataFrame(rows)

# Inverse scaling to original scale
preds_ml = inverse_scale_df(preds_ml_scaled, ["GlobalML", "ClusterML", "LocalML"])
preds_ml.head()

,id,ds,GlobalML,ClusterML,LocalML
0,Q14052,2004-06-30 23:59:59.999999999,3161.249026,3112.048254,2967.043139
1,Q14052,2004-09-30 23:59:59.999999999,3089.091933,2953.404029,2853.014223
2,Q14052,2004-12-31 23:59:59.999999999,3024.058113,2833.748192,2836.645186
3,Q14052,2005-03-31 23:59:59.999999999,2962.862384,2805.249759,2883.664201
4,Q14052,2005-06-30 23:59:59.999999999,2914.133373,2829.925198,2938.365278


### 5.3. Подсчет метрик и сравнение с baseline

In [10]:
ml_models = ["GlobalML", "ClusterML", "LocalML"]

ml_eval = test_long.merge(preds_ml, on=["id", "ds"], how="inner")
smape_ml = smape(ml_eval, models=ml_models, id_col="id", target_col="y")

summary_ml = pd.DataFrame({
    "sMAPE": smape_ml[ml_models].mean(),
}).sort_values("sMAPE")

summary_all = pd.concat(
    [
        summary_base.rename_axis("model"),
        summary_ml.rename_axis("model")
    ],
    axis=0
)

summary_all.sort_values("sMAPE")

,sMAPE
model,
AutoETS,0.036463
AutoTheta,0.036586
Naive,0.050769
SeasonalNaive,0.061119
ClusterML,0.072825
LocalML,0.088672
GlobalML,0.097452


**Вывод:** локальные модели действительно проиграли глобальным моделям, обученным на кластерах похожих рядов

## 6. Визуализация предсказаний

In [11]:
def plot_forecast_interactive(series_id, train_long, test_long, preds_base, preds_ml):
    df_train = train_long[train_long["id"] == series_id]
    df_test = test_long[test_long["id"] == series_id]

    df_base = preds_base[preds_base["id"] == series_id]
    df_ml = preds_ml[preds_ml["id"] == series_id]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_train["ds"], y=df_train["y"], mode="lines", name="train"))
    fig.add_trace(go.Scatter(x=df_test["ds"], y=df_test["y"], mode="lines", name="test"))

    # Baseline models
    for m in ["Naive", "SeasonalNaive", "AutoTheta", "AutoETS"]:
        if m in df_base.columns:
            fig.add_trace(go.Scatter(x=df_base["ds"], y=df_base[m], mode="lines", name=m))

    # ML models
    for m in ["GlobalML", "ClusterML", "LocalML"]:
        if m in df_ml.columns:
            fig.add_trace(go.Scatter(x=df_ml["ds"], y=df_ml[m], mode="lines", name=m))

    fig.update_layout(title=f"Series {series_id}", xaxis_title="ds", yaxis_title="y")
    fig.show()


# plot a few random series
example_ids = rng.choice(train_long["id"].unique(), size=3, replace=False)
for sid in example_ids:
    plot_forecast_interactive(sid, train_long, test_long, preds_base, preds_ml)
